In [0]:
import pandas as pd

ruta = '/Volumes/workspace/default/kmc_data/kmc_dataset_procesado_completo.csv'
df = pd.read_csv(ruta, low_memory=False)
print(f"¡Éxito! Dimensiones: {df.shape}")

In [0]:
# ── CELDA 1: CONFIGURACIÓN Y LIBRERÍAS ──────────────────────────────────────
import os
import warnings
import pandas as pd
import numpy as np
import logging
from scipy.stats import chi2_contingency, mannwhitneyu

# Optimizaciones para Databricks
os.environ["OMP_NUM_THREADS"] = "1"
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", module="threadpoolctl")

# Configuración del Logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(message)s'
)

def sep(title='', width=76):
    t = title.strip()
    if t:
        logging.info(f"\n{'─'*4} {t} {'─'*max(2, width-len(t)-6)}")
    else:
        logging.info('─' * width)

print("✅ Celda 1 ejecutada: Entorno configurado correctamente.")

In [0]:
# ── CELDA 2: CARGA Y FUSIÓN DE DATOS ────────────────────────────────────────

# Rutas de los archivos
PATH_ORIGINAL = '/Volumes/workspace/default/kmc_data/kmc_dataset_procesado_completo.csv'
PATH_CLUSTERS = '/Volumes/workspace/default/kmc_data/clusters_GOi4.csv'

def cargar_y_fusionar():
    sep('CARGA DE DATOS (UNITY CATALOG)')
    
    # 1. Carga del dataset original (2.267 variables)
    logging.info("Leyendo sábana original de datos...")
    df_orig = pd.read_csv(PATH_ORIGINAL, low_memory=False)
    logging.info(f"✓ Datos originales cargados: {df_orig.shape[0]} pacientes x {df_orig.shape[1]} variables")
    
    # 2. Carga de los resultados del clustering
    logging.info("Leyendo etiquetas del modelo no supervisado...")
    df_clust = pd.read_csv(PATH_CLUSTERS)
    
    # 3. Fusión (Merge) usando la columna 'code'
    if 'code' in df_orig.columns and 'code' in df_clust.columns:
        df_merged = pd.merge(df_orig, df_clust[['code', 'GO_i', 'GO_i_label', 'sil_sample']], on='code', how='inner')
        logging.info(f"✓ Fusión exitosa. Dataset final listo para validación: {df_merged.shape}")
        return df_merged
    else:
        logging.error("❌ No se encontró la columna 'code' para hacer la fusión. Verifica los nombres.")
        return None

df = cargar_y_fusionar()

# Mostramos un resumen rápido para confirmar
if df is not None:
    display(df[['code', 'GO_i_label', 'sil_sample']].head())

In [0]:
# ── CELDA 3: VALIDACIÓN DE CONFUSORES SOCIODEMOGRÁFICOS ─────────────────────

def analisis_confusores(dataset):
    sep('VALIDACIÓN 1: CONFUSORES SOCIODEMOGRÁFICOS')
    
    vars_socio = {
        'BIRTH_sexo5': 'categórica', 
        'RT_Family_SES': 'numérica' # El SES suele ser un índice continuo o suma de puntajes
    }
    
    for var, tipo in vars_socio.items():
        if var not in dataset.columns:
            logging.warning(f"  [!] Variable '{var}' no encontrada. Revisa el nombre exacto.")
            continue
            
        logging.info(f"\n➤ Analizando: {var.upper()}")
        
        # Validación para variables Categóricas (Sexo, Estrato) -> Chi-Cuadrado
        if tipo == 'categórica':
            ct = pd.crosstab(dataset['GO_i'], dataset[var])
            _, p, _, _ = chi2_contingency(ct)
            sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
            
            # Mostramos cómo se distribuyen los porcentajes
            ct_pct = pd.crosstab(dataset['GO_i'], dataset[var], normalize='index') * 100
            for go in [1, 2]:
                vals = " | ".join([f"{val}:{ct_pct.loc[go, val]:.1f}%" for val in ct_pct.columns])
                logging.info(f"    GO-{go}: [ {vals} ]")
            logging.info(f"    p-value (Chi2): {p:.4f} {sig}")
            
        # Validación para variables Numéricas (Años de educación) -> Mann-Whitney
        elif tipo == 'numérica':
            v1 = dataset.loc[dataset['GO_i']==1, var].dropna()
            v2 = dataset.loc[dataset['GO_i']==2, var].dropna()
            _, p = mannwhitneyu(v1, v2)
            sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
            
            logging.info(f"    GO-1 (Alto) Media: {v1.mean():.1f} ± {v1.std():.1f}")
            logging.info(f"    GO-2 (Bajo) Media: {v2.mean():.1f} ± {v2.std():.1f}")
            logging.info(f"    p-value (M-W): {p:.4f} {sig}")

if df is not None:
    analisis_confusores(df)

In [0]:
# ── CELDA 4: ANÁLISIS DE PACIENTES FRONTERA ─────────────────────────────────

def analizar_frontera(dataset):
    sep('VALIDACIÓN 2: PACIENTES EN ZONA GRIS (FRONTERA)')
    
    # Pacientes con Silhouette entre -0.05 y 0.05
    mask_frontera = (dataset['sil_sample'] >= -0.05) & (dataset['sil_sample'] <= 0.05)
    n_frontera = mask_frontera.sum()
    
    logging.info(f"Pacientes en frontera (Silhouette ~0): {n_frontera} ({n_frontera/len(dataset)*100:.1f}%)")
    
    df_frontera = dataset.loc[mask_frontera]
    
    # Comprobamos de qué grupo perinatal vienen (Canguro, Control, Referencia)
    # ⚠️ Asegúrate de que 'ubicac' sea el nombre de la variable de grupo original ⚠️
    if 'ubicac' in df_frontera.columns:
        dist_nacimiento = df_frontera['ubicac'].value_counts()
        logging.info("\nOrigen de estos pacientes frontera:")
        for g, n in dist_nacimiento.items():
            # Mapeo según tu documentación (Ajusta si los números son otros)
            nombre_g = {1: 'KMC (Canguro)', 2: 'TC (Control)', 3: 'Referencia (Término)'}.get(g, f"Grupo {g}")
            logging.info(f"  ➤ {nombre_g}: {n} pacientes")
    else:
        logging.warning("No se encontró la variable 'ubicac' para ver el grupo de origen.")

if df is not None:
    analizar_frontera(df)